# Interview Prep: 21 - Full-Stack System Design (Telemedicine Case Study)

System design interviews often focus on a single use case to test your depth across the stack. This notebook uses a **Telemedicine Platform** to explore the nuances of Web vs. Mobile and real-time synchronization.

---

## 1. Web vs. Mobile: Architectural Nuances

### **Interview Scenario**: "We are building a Telemedicine app. How does the architecture change if a doctor is on a Web dashboard vs. a patient on a Mobile app?"

**The Contrast**:
- **Web (Doctor)**: Focus on complex data visualization, dashboarding, and SEO (if public-facing). Architecture: Standard REST/GraphQL over HTTP. High bandwidth assumed.
- **Mobile (Patient)**: Focus on low-latency interactions, push notifications, and **offline resilience**. Architecture: Offline-first sync engines, background workers, and platform-specific APIs (Camera, HealthKit).

---

## 2. Real-time Communication: WebSockets vs. WebRTC

### **Interview Question**: "When would you use WebRTC instead of WebSockets for a Telemedicine app?"

**Answer**:
- **WebSockets**: Best for **text-based chat**, status updates (Doctor is typing...), and lightweight data syncing. It uses a server as a proxy.
- **WebRTC**: Best for **Video/Audio calls**. It is Peer-to-Peer (P2P), meaning data flows directly between the patient and doctor, reducing latency and server load.

---

## 3. Offline-First & Sync Engines

### **Interview Question**: "How do you handle a patient updating their symptoms while in an area with no internet connection?"

**The Solution**: An **Offline-First Sync Engine** (e.g., WatermelonDB, SQLite with a sync layer).
1. **Local Write**: The app writes the data to a local DB immediately.
2. **Sync Queue**: A background worker waits for network connectivity.
3. **Conflict Resolution**: When back online, the app pushes the change. We use **Last Writer Wins** or **CRDTs (Conflict-free Replicated Data Types)** to merge data if the doctor modified the record at the same time.

---

## 4. Push Notification Infrastructure

### **Interview Question**: "How do you reliably send a push notification to millions of mobile users at once?"

**The Architecture**:
- **System**: Backend (Python/Go) -> Message Queue (Kafka/SQS) -> Notification Service -> FCM (Firebase) / APNs (Apple).
- **Scalability**: Decouple the "send" request from the actual delivery using worker nodes. Ensure the device tokens are stored and refreshed in a highly available DB (Redis/DynamoDB).

---

## 5. Coding Challenge: Designing the Sync Logic

**Task**: Write the pseudocode/logic for a background sync worker that handles exponential backoff for a failed data push.


In [ ]:
import time
import random

def sync_data(attempt=1, max_attempts=5):
    try:
        print(f"Attempting to sync data (Attempt {attempt})...")
        # Simulate network check and push
        if random.random() < 0.6: # 60% failure chance
             raise ConnectionError("Network Unstable")
        print("Sync Successful!")
        return True
    except ConnectionError as e:
        if attempt >= max_attempts:
            print("Max attempts reached. Stopping.")
            return False
        
        # Exponential backoff with jitter
        wait_time = (2 ** attempt) + random.uniform(0, 1)
        print(f"Sync Failed: {e}. Waiting {wait_time:.2f}s...")
        time.sleep(wait_time)
        return sync_data(attempt + 1)

sync_data()